<a href="https://colab.research.google.com/github/Brandon9010/Brandon9010/blob/main/Supply_Chaine_Routing/Supply_Chain_Optimization_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Supply Chain Logistics Optimization Project
By: Brandon Frosch

###Set-up & Library Installation

In [9]:
#Instal Libraries
import sys
import os

# Install Pyomo and solvers if running in Google Colab
if 'google.colab' in sys.modules:
    !pip install idaes-pse --pre
    !pip install kagglehub
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

from pyomo.environ import *
import pandas as pd
import kagglehub

# =============

Getting files...
Done
-----------------------------------------------------------------
IDAES Extensions Build Versions
Solvers:  v3.4.2 20240811 ubuntu2204-x86_64
Library:  v3.4.2 20240811 ubuntu2204-x86_64



##Download and Load Excel Files

In [19]:
print("Downloading dataset from Kaggle...")
dataset_path = kagglehub.dataset_download("anisseezzebdi/supply-chain-logistics-problem")

# Find the specific Excel file in the downloaded folder
excel_file = None
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith('.xlsx'):
            excel_file = os.path.join(root, file)
            break

if not excel_file:
    raise FileNotFoundError("Could not find the Excel file in the downloaded Kaggle dataset.")

print(f"Reading sheets from: {os.path.basename(excel_file)}...")

# Load all 7 sheets directly from the workbook
excel_data = pd.ExcelFile(excel_file)
orders = excel_data.parse('OrderList')

#Clean Order ID
orders['Order ID'] = orders['Order ID'].apply(lambda x: str(int(x)) if pd.notnull(x) else x)

freight = excel_data.parse('FreightRates')
wh_costs = excel_data.parse('WhCosts')
wh_caps = excel_data.parse('WhCapacities')
products = excel_data.parse('ProductsPerPlant')
vmi = excel_data.parse('VmiCustomers')
plant_ports = excel_data.parse('PlantPorts')

Using Colab cache for faster access to the 'supply-chain-logistics-problem' dataset.
Reading sheets from: Supply chain logistics problem.xlsx...


##Data Engineering: Feasible Paths & Costing

In [20]:
print("Processing multi-echelon network constraints...")

# Clean orders dataframe to avoid column conflicts if Plant Code exists
if 'Plant Code' in orders.columns:
    orders = orders.drop(columns=['Plant Code'])

# Identify which Plant can supply which Order based on Product ID
df = orders.merge(products, on='Product ID', how='inner')

# Join with Ports used by each Plant
df = df.merge(plant_ports, on='Plant Code', how='inner')

# Calculate the "Storage Cost" for each assignment
df = df.merge(wh_costs, left_on='Plant Code', right_on='WH', how='left')

# Join with Freight Rates to find shipping lanes (Origin Port to Destination Port)
freight_map = freight.groupby(['orig_port_cd', 'dest_port_cd'])['rate'].mean().reset_index()
df = df.merge(
    freight_map,
    left_on=['Port', 'Destination Port'],
    right_on=['orig_port_cd', 'dest_port_cd'],
    how='inner'
)

# Calculate Total Landed Cost per possible assignment
df['Landed_Cost'] = (df['Weight'] * df['rate']) + (df['Unit quantity'] * df['Cost/unit'])

# 1. Find products that are stocked in more than 1 plant
plant_counts = products.groupby('Product ID')['Plant Code'].nunique()
multi_plant_products = plant_counts[plant_counts > 1].index.tolist()

# 2. Filter our dataframe to ONLY include orders for these flexible products
flexible_orders_df = df[df['Product ID'].isin(multi_plant_products)]

# 3. Take 50 unique orders from this highly-constrained subset
sample_orders = flexible_orders_df['Order ID'].unique()[:50]
df_subset = df[df['Order ID'].isin(sample_orders)].copy()

Processing multi-echelon network constraints...


##Create Dictionary for Pyomo

In [21]:
#Create Dictionary
possible_assignments = {}
for i, row in df_subset.iterrows():
    possible_assignments[(row['Order ID'], row['Plant Code'])] = {
        'cost': row['Landed_Cost'],
        'units': row['Unit quantity']
    }

order_ids = df_subset['Order ID'].unique()
plant_ids = df_subset['Plant Code'].unique()

# Handle trailing space in Kaggle's 'Daily Capacity ' column
capacity_col = 'Daily Capacity ' if 'Daily Capacity ' in wh_caps.columns else 'Daily Capacity'
plant_capacities = wh_caps.set_index('Plant ID')[capacity_col].to_dict()

##Build Pyomo Model

In [22]:
#Initialize Model
print("Building Linear Programming Model...")
model = ConcreteModel()

# Sets
model.ORDERS = Set(initialize=order_ids)
model.PLANTS = Set(initialize=plant_ids)

# Variables: Binary
model.x = Var(possible_assignments.keys(), domain=Binary)

Building Linear Programming Model...


##Objective: Minimize Total Distribution Cost

In [23]:
#Initialize Objective
model.obj = Objective(
    expr=sum(
        model.x[o, p] * possible_assignments[(o, p)]['cost']
        for (o, p) in possible_assignments.keys()
    ),
    sense=minimize
)

##Model Constraints

In [24]:
#Initialize Constraints
model.constraints = ConstraintList()

# A. Fulfillment Constraint
for o in model.ORDERS:
    valid_plants = [p for p in model.PLANTS if (o, p) in possible_assignments]
    if valid_plants:
        model.constraints.add(sum(model.x[o, p] for p in valid_plants) == 1)

# B. Warehouse Capacity Constraint
for p in model.PLANTS:
    valid_orders = [o for o in model.ORDERS if (o, p) in possible_assignments]
    if valid_orders:
        model.constraints.add(
            sum(model.x[o, p] for o in valid_orders) <= plant_capacities.get(p, float('inf'))
        )

##Solve and Output Distribution Plan

In [25]:
#Solve
print("Solving...")
solver = SolverFactory("cbc")
result = solver.solve(model, tee=False)

#Output
print("\n--- Optimization Complete ---")
print("Status:", result.solver.status)
print("Minimum Total Logistics Cost: $", round(value(model.obj), 2))

plan = []
for (o, p) in possible_assignments.keys():
    if value(model.x[o, p]) > 0.5:
        plan.append({
            'Order_ID': o,
            'Assigned_Plant': p,
            'Cost': round(possible_assignments[(o, p)]['cost'], 2)
        })

results_df = pd.DataFrame(plan)
print("\nOptimal Shipping Plan (First 10 Orders):")
print(results_df.head(10))


Solving...

--- Optimization Complete ---
Status: ok
Minimum Total Logistics Cost: $ 51158.11

Optimal Shipping Plan (First 10 Orders):
     Order_ID Assigned_Plant     Cost
0  1447231067        PLANT09  2397.88
1  1447306538        PLANT09  2357.42
2  1447355121        PLANT03   960.88
3  1447240887        PLANT09  4634.00
4  1447364831        PLANT03  1802.74
5  1447240888        PLANT09  2399.28
6  1447352523        PLANT09  1653.78
7  1447182397        PLANT03   956.22
8  1447278496        PLANT09  3867.11
9  1447255583        PLANT03  1802.22
